### Character RNN

#### Creating the Training Dataset

In [1]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [2]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [3]:
# Build the vocabulary
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [4]:
# encode - decode
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}
print(char_to_id["a"])
print(id_to_char[13])


13
a


In [5]:
import torch

def encode_text(text: str):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids: torch.Tensor):
    return "".join([id_to_char[char_id.item()] for char_id in char_ids])

In [6]:
encoded = encode_text("Hellow, world!")
print(encoded)

decode_text(encoded)

tensor([20, 17, 24, 24, 27, 35,  6,  1, 35, 27, 30, 24, 16,  2])


'hellow, world!'

In [7]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text: str, window_length: int):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length

    def __getitem__(self, idx: int):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx : end]
        target = self.encoded_text[idx + 1 : end + 1]
        return window, target

In [8]:
window_length = 50
batch_size = 512  # reduce if your GPU cannot handle such a large batch size
train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

#### Embeddings

In [10]:
import torch.nn as nn

torch.manual_seed(42)
embed = nn.Embedding(5, 3)    # 5 categories x 3D embeddings
embed(torch.tensor([[3, 2], [0, 2]]))


tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

In [11]:
embed.weight

Parameter containing:
tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617],
        [ 0.2674,  0.5349,  0.8094],
        [ 1.1103, -1.6898, -0.9890]], requires_grad=True)

In [13]:
# An embedding layer is equivalent to one-hot encoding
# followed by a linear layer (with no bias)
lin = nn.Linear(5, 3, bias=False)
lin(torch.tensor([[0., 0., 0., 1., 0.]]))

tensor([[ 0.2764,  0.1202, -0.1955]], grad_fn=<MmBackward0>)

In [15]:
lin.weight.T

tensor([[ 0.3453,  0.3613,  0.1882],
        [ 0.0744,  0.0489,  0.3993],
        [-0.1452, -0.1410,  0.2585],
        [ 0.2764,  0.1202, -0.1955],
        [ 0.0697, -0.1213,  0.2582]], grad_fn=<PermuteBackward0>)

#### Building and Training the Char-RNN Model

In [16]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size:int, n_layers: int = 2, embed_dim: int = 10,
                 hidden_dim: int = 128, dropout: float = 0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

In [17]:
def evaluate_tm(model, data_loader, metric, device=None):
    if device:
        model = model.to(device)
        metric = metric.to(device)
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)  # update it at each iteration
    return metric.compute()  # compute the final result at the end


def train_with_early_stopping(model, optimizer, criterion, metric, train_loader, valid_loader,
               n_epochs, early_stopping=None, scheduler=None, device=None):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    if device:
        model = model.to(device)
        metric = metric.to(device)
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            if device:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)    # forward pass
            loss = criterion(y_pred, y_batch)    # compute loss for this batch
            total_loss = total_loss +  loss.item()
            loss.backward()    # compute gradients of the loss
            optimizer.step()   # adjust the model weights based on the gradients
            optimizer.zero_grad()    # zero the gradients for next batch
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        val_metric = evaluate_tm(model, valid_loader, metric, device=device).item()
        history["valid_metrics"].append(val_metric)
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
        # Check early stopping condition
        if early_stopping:
            early_stopping.check_early_stop(val_metric)
            if early_stopping.stop_training:
                print(f"Early stopping at epoch {epoch}")
                break
        if scheduler:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_metric)
            else:
                scheduler.step()
            print(f"Learning rate: {scheduler.get_last_lr()[0]:.5f}")
    return history

In [20]:
import torchmetrics

model = ShakespeareModel(len(vocab))

device = "mps"
n_epochs = 20
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)

history = train_with_early_stopping(model, optimizer, xentropy, accuracy, train_loader, valid_loader,
                n_epochs, device=device)

Epoch 1/20, train loss: 1.6193, train metric: 0.5096, valid metric: 0.5076
Epoch 2/20, train loss: 1.3900, train metric: 0.5659, valid metric: 0.5286
Epoch 3/20, train loss: 1.3587, train metric: 0.5734, valid metric: 0.5336
Epoch 4/20, train loss: 1.3437, train metric: 0.5771, valid metric: 0.5366
Epoch 5/20, train loss: 1.3350, train metric: 0.5792, valid metric: 0.5401
Epoch 6/20, train loss: 1.3293, train metric: 0.5807, valid metric: 0.5447
Epoch 7/20, train loss: 1.3251, train metric: 0.5818, valid metric: 0.5404
Epoch 8/20, train loss: 1.3217, train metric: 0.5827, valid metric: 0.5463
Epoch 9/20, train loss: 1.3193, train metric: 0.5833, valid metric: 0.5437
Epoch 10/20, train loss: 1.3174, train metric: 0.5838, valid metric: 0.5422
Epoch 11/20, train loss: 1.3155, train metric: 0.5842, valid metric: 0.5446
Epoch 12/20, train loss: 1.3144, train metric: 0.5845, valid metric: 0.5422
Epoch 13/20, train loss: 1.3130, train metric: 0.5848, valid metric: 0.5450
Epoch 14/20, train lo

In [27]:
torch.save(model.state_dict(), "my_shakespeare_model.pt")

In [22]:
model.eval()
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)  # shape: [batch_size, vocab_size, window_size]
    predicted_char_id = Y_logits[0, :, -1].argmax().item()  # we care only for the last output
    predicted_char = id_to_char[predicted_char_id]
predicted_char

'e'

In [28]:
Y_logits[0, :, -1].shape

torch.Size([39])

#### Generating Fake Shakespearean Text

In [26]:
torch.manual_seed(42)
probs = torch.tensor([[0.5, 0.4, 0.1]])  # probas = 50%, 40%, 10%
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [34]:
import torch.nn.functional as F

# instead of choosing the most probable character every time,
# sample the next character according to the probability distribution
def next_char(model: nn.Module, text: str, temperature: int | float = 1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[predicted_char_id]

In [ ]:
def extend_text(model: nn.Module, text: str, n_chars: int = 80, temperature: int | float = 1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)
    return text

In [35]:
print(extend_text(model, "To be or not to b", temperature=0.01))

To be or not to be the seat
and see the consuls and stand all the senate and state and state and 


In [36]:
print(extend_text(model, "To be or not to b", temperature=0.4))

To be or not to be a bark,
and say 'tis best were a day and so late,
and then i will not see the 


In [37]:
print(extend_text(model, "To be or not to b", temperature=100))

To be or not to b lkndob3pr? unl!.dt3 puzoyvmlfdhmf,h
jc,3o;ai?gvg!cdnykf
kucecovsiy?p&x.'wmkmyzj


In [39]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

In [41]:
del_vars(["accuracy", "embed", "encoded", "encoded_text", "optimizer", "probs",
          "samples", "x", "y", "shakespeare_text", "stateful_test_loader",
          "stateful_train_loader", "Y_logits", "stateful_valid_loader",
          "test_loader", "train_loader", "valid_loader", "xentropy"])

In [42]:
Out.clear()

### Sentiment Analysis Using Hugging Face Libraries

In [43]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]

README.md: 0.00B [00:00, ?B/s]

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [44]:
imdb_train_set[1]["text"]

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

In [45]:
imdb_train_set[1]["label"]

1

In [46]:
imdb_train_set[16]["text"]

"Lillian Hellman's play, adapted by Dashiell Hammett with help from Hellman, becomes a curious project to come out of gritty Warner Bros. Paul Lukas, reprising his Broadway role and winning the Best Actor Oscar, plays an anti-Nazi German underground leader fighting the Fascists, dragging his American wife and three children all over Europe before finding refuge in the States (via the Mexico border). They settle in Washington with the wife's wealthy mother and brother, though a boarder residing in the manor is immediately suspicious of the newcomers and spends an awful lot of time down at the German Embassy playing poker. It seems to take forever for this drama to find its focus, and when we realize what the heart of the material is (the wise, honest, direct refugees teaching the clueless, head-in-the-sand Americans how the world has suddenly changed), it seems a little patronizing--the viewer is quite literally put in the relatives' place, being lectured to. Lukas has several speeches 

In [47]:
imdb_train_set[16]["label"]

0

#### Tokenization Using the Hugging Face Tokenizers Library

In [48]:
# Byte Pair Encoding
import tokenizers

bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
train_reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)


In [49]:
some_review = "what an awesome movie! 😊"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [50]:
bpe_encoding.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']

In [51]:
bpe_token_ids = bpe_encoding.ids
bpe_token_ids

[303, 139, 373, 149, 240, 211, 4, 1]

In [52]:
bpe_tokenizer.get_vocab()

{'ue': 850,
 'story': 349,
 'super': 858,
 '^': 39,
 'ble': 376,
 'from': 283,
 'horror': 716,
 '₤': 135,
 'howe': 744,
 'bud': 955,
 '\\': 37,
 'ough': 305,
 'guy': 652,
 'even': 347,
 'na': 864,
 'up': 288,
 'ave': 568,
 '#': 6,
 'such': 589,
 'always': 787,
 'ou': 147,
 'mind': 731,
 'jo': 341,
 'hi': 537,
 'ter': 187,
 'sp': 348,
 'find': 571,
 'vel': 779,
 'her': 233,
 'ably': 604,
 'age': 402,
 'only': 364,
 ').': 521,
 'hel': 741,
 'mor': 933,
 '\x08': 2,
 'were': 382,
 'ess': 266,
 'epis': 931,
 'know': 463,
 'ents': 649,
 'one': 205,
 'in': 137,
 'dire': 426,
 '¾': 96,
 'mem': 651,
 'poss': 886,
 '«': 87,
 'led': 813,
 'ú': 124,
 'hol': 895,
 'role': 806,
 '\x9a': 78,
 'ward': 772,
 'is': 141,
 'ment': 323,
 'ph': 618,
 'main': 783,
 'kind': 756,
 'body': 953,
 'used': 810,
 'ger': 844,
 'times': 650,
 'black': 970,
 'sc': 238,
 'looking': 976,
 'americ': 784,
 'u': 62,
 '\x97': 77,
 'ha': 208,
 'ind': 312,
 'dif': 694,
 'ical': 475,
 'sense': 939,
 'ating': 621,
 'ace': 457,


In [53]:
bpe_tokenizer.token_to_id("<unk>")

1

In [54]:
bpe_tokenizer.id_to_token(1)

'<unk>'

In [58]:
bpe_tokenizer.id_to_token(0)

'<pad>'

In [55]:
bpe_tokenizer.decode(bpe_token_ids)

'what an aw es ome movie !'

In [56]:
bpe_encoding.offsets

[(0, 4), (5, 7), (8, 10), (10, 12), (12, 15), (16, 21), (21, 22), (23, 24)]

In [57]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [59]:
# harmonize the length of the different reviews
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [60]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [61]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [62]:
lengths = attention_mask.sum(dim=1)
lengths

tensor([281, 114, 285])

In [66]:
## Byte-level BPE
bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
special_tokens = ["<pad>", "<unk>"]
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(train_reviews, bbpe_trainer)

In [67]:
bbpe_encoding = bbpe_tokenizer.encode(some_review)
bbpe_encoding

Encoding(num_tokens=12, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [70]:
bbpe_decoded = bbpe_tokenizer.decode(bbpe_encoding.ids)
bbpe_decoded.replace(" ", "").replace("Ġ", " ").strip()

'what an awesome movie! Łĺ'

In [80]:
# WordPiece
wp_model = tokenizers.models.WordPiece(unk_token="<unk>")
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
wp_trainer = tokenizers.trainers.WordPieceTrainer(vocab_size=1000, special_tokens=special_tokens)
wp_tokenizer.train_from_iterator(train_reviews, wp_trainer)

In [81]:
wp_encodings = wp_tokenizer.encode(some_review)
wp_encodings

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [82]:
wp_decoded = wp_tokenizer.decode(wp_encodings.ids)
wp_decoded

'what an aw ##es ##ome movie !'

In [83]:
wp_decoded.replace(" ##", "")

'what an awesome movie !'

#### Reusing Pretrained Tokenizers

In [84]:
import transformers

gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True, max_length=500)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [85]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [86]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

In [90]:
gpt2_tokenizer.convert_ids_to_tokens(gpt2_token_ids)

['stage',
 'Ġadaptations',
 'Ġoften',
 'Ġhave',
 'Ġa',
 'Ġmajor',
 'Ġfault',
 '.',
 'Ġthey',
 'Ġoften']

In [91]:
# WordPiece pretrained tokenizer
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True, truncation=True,
                               max_length=500, return_tensors="pt")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [92]:
bert_encoding["input_ids"]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [93]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [94]:
# Unigral LM pretrained tokenizer
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_encoding = albert_tokenizer(train_reviews[:3], padding=True, truncation=True,
                               max_length=500, return_tensors="pt")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

In [96]:
albert_encoding["input_ids"]

tensor([[    2,   876,  5004,    18,   478,    57,    21,   394,  4173,     9,
            59,   478,   340,    70,   699,   101,    21,   171,  3336,    23,
          1659,  1037,    27,    14,   876,    13,     5,  4289,    28,    13,
             7,  4893,   449,     7,     6,     9, 12508,  1612,  5909,    22,
            18,  1400,  8968,    14,   171,  2481,    15,    56,    25,  1118,
          1956,   179,    14,  2151,  1434,    61,    90,   683,  2404,     9,
           174,    15,    32,    22,    18,  2210,    20,   361,    35,    26,
            98,    32,    25,     9,    14,  5427,   128,   832, 22427,    17,
          4479, 24604,    25,  1450,  7472,     9,    14, 12289,    16,    66,
          1429,    50, 12891,     9, 22427,    25, 10356,    28,   550,    15,
            17, 24604,  3049,    53,    16,    33,   310, 11285,    20,   510,
           601,     9,     1,  5145,    13,   118,     1,  5145,    13,   118,
             1,    49, 14586,    30,    31,    22,  

In [97]:
# wrap a tokenizer from Tokenizers library into an object with the Transformers API
hf_tokenizer = transformers.PreTrainedTokenizerFast(tokenizer_object=bpe_tokenizer)
hf_encodings = hf_tokenizer(train_reviews[:3], padding=True, truncation=True,
                               max_length=500, return_tensors="pt")

#### Building and Training a Sentiment Analysis Model

In [108]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [[review["label"]] for review in batch]
    encodings = tokenizer(reviews, padding=True, truncation=True,
                          max_length=200, return_tensors="pt")
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels

batch_size = 256
imbdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size,
                                collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set, batch_size=batch_size, collate_fn=collate_fn)

In [109]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
                 pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])  # or outputs[:, -1] (outputs and hidden states are equal in GRU)

In [110]:
torch.manual_seed(42)
vocab_size = bert_tokenizer.vocab_size
model = SentimentAnalysisModel(vocab_size)
device = "mps"
n_epochs = 20
bce_loss = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters())
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train_with_early_stopping(model, optimizer, bce_loss, accuracy, imbdb_train_loader, imdb_valid_loader,
                n_epochs, device=device)

Epoch 1/20, train loss: 0.6905, train metric: 0.5304, valid metric: 0.5284
Epoch 2/20, train loss: 0.6316, train metric: 0.6434, valid metric: 0.6082
Epoch 3/20, train loss: 0.4105, train metric: 0.8176, valid metric: 0.8194
Epoch 4/20, train loss: 0.2885, train metric: 0.8834, valid metric: 0.8472
Epoch 5/20, train loss: 0.2036, train metric: 0.9230, valid metric: 0.8312
Epoch 6/20, train loss: 0.1341, train metric: 0.9542, valid metric: 0.8610
Epoch 7/20, train loss: 0.0933, train metric: 0.9703, valid metric: 0.8560
Epoch 8/20, train loss: 0.0425, train metric: 0.9893, valid metric: 0.8572
Epoch 9/20, train loss: 0.0273, train metric: 0.9939, valid metric: 0.8482
Epoch 10/20, train loss: 0.0654, train metric: 0.9816, valid metric: 0.8508
Epoch 11/20, train loss: 0.0182, train metric: 0.9958, valid metric: 0.8476
Epoch 12/20, train loss: 0.0091, train metric: 0.9984, valid metric: 0.8492
Epoch 13/20, train loss: 0.0105, train metric: 0.9977, valid metric: 0.8476
Epoch 14/20, train lo

#### Bidirectional RNNs

In [112]:
class BDSentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
                 pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [113]:
torch.manual_seed(42)
model = BDSentimentAnalysisModel(vocab_size)
n_epochs = 20
bce_loss = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters())
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train_with_early_stopping(model, optimizer, bce_loss, accuracy, imbdb_train_loader, imdb_valid_loader,
                n_epochs, device=device)

Epoch 1/20, train loss: 0.6647, train metric: 0.5936, valid metric: 0.6668
Epoch 2/20, train loss: 0.5477, train metric: 0.7255, valid metric: 0.7726
Epoch 3/20, train loss: 0.4036, train metric: 0.8207, valid metric: 0.7432
Epoch 4/20, train loss: 0.2963, train metric: 0.8798, valid metric: 0.7532
Epoch 5/20, train loss: 0.2173, train metric: 0.9151, valid metric: 0.8050
Epoch 6/20, train loss: 0.1448, train metric: 0.9474, valid metric: 0.8398
Epoch 7/20, train loss: 0.1032, train metric: 0.9654, valid metric: 0.8280
Epoch 8/20, train loss: 0.0499, train metric: 0.9858, valid metric: 0.8224
Epoch 9/20, train loss: 0.0242, train metric: 0.9945, valid metric: 0.8148
Epoch 10/20, train loss: 0.0527, train metric: 0.9850, valid metric: 0.8222
Epoch 11/20, train loss: 0.0153, train metric: 0.9966, valid metric: 0.8388
Epoch 12/20, train loss: 0.0619, train metric: 0.9841, valid metric: 0.8162
Epoch 13/20, train loss: 0.0244, train metric: 0.9929, valid metric: 0.8342
Epoch 14/20, train lo

In [115]:
Out.clear()  # clear Jupyter's `Out` variable which saves all the cell outputs
del_vars(["albert_token_ids", "attention_mask", "bpe_batch_ids", "encoded_text",
          "lengths", "optimizer", "padded", "probs", "samples", "sequences",
          "x", "xentropy", "y", "Y_logits"])

#### Reusing Pretrained Embeddings and Language Models

In [114]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
bert_model.embeddings.word_embeddings

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Embedding(30522, 768, padding_idx=0)

In [116]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64,
                 dropout=0.2):
        super().__init__()
        weights = pretrained_embeddings.weight.data
        self.embed = nn.Embedding.from_pretrained(weights, freeze=True)
        embed_dim = weights.shape[-1]
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [117]:
torch.manual_seed(42)
pretrained_embeddings = bert_model.embeddings.word_embeddings
model = SentimentAnalysisModelPreEmbeds(pretrained_embeddings)
n_epochs = 17
bce_loss = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(model.parameters())
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=2, factor=0.5)
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train_with_early_stopping(model, optimizer, bce_loss, accuracy, imbdb_train_loader, imdb_valid_loader,
                n_epochs, device=device)

Epoch 1/17, train loss: 0.6879, train metric: 0.5537, valid metric: 0.6108
Epoch 2/17, train loss: 0.6459, train metric: 0.6258, valid metric: 0.6256
Epoch 3/17, train loss: 0.5812, train metric: 0.6954, valid metric: 0.7264
Epoch 4/17, train loss: 0.5057, train metric: 0.7563, valid metric: 0.6392
Epoch 5/17, train loss: 0.4152, train metric: 0.8092, valid metric: 0.7906
Epoch 6/17, train loss: 0.3606, train metric: 0.8375, valid metric: 0.8262
Epoch 7/17, train loss: 0.3263, train metric: 0.8554, valid metric: 0.8474
Epoch 8/17, train loss: 0.2988, train metric: 0.8711, valid metric: 0.8242
Epoch 9/17, train loss: 0.2727, train metric: 0.8878, valid metric: 0.7330
Epoch 10/17, train loss: 0.2515, train metric: 0.8964, valid metric: 0.7380
Epoch 11/17, train loss: 0.2422, train metric: 0.9007, valid metric: 0.8212
Epoch 12/17, train loss: 0.2006, train metric: 0.9201, valid metric: 0.8504
Epoch 13/17, train loss: 0.1752, train metric: 0.9299, valid metric: 0.8574
Epoch 14/17, train lo

In [119]:
# unfreeze the embeddings to fine-tune them for a few epochs
model.embed.weight.requires_grad = True
history_cont = train_with_early_stopping(model, optimizer, bce_loss, accuracy, imbdb_train_loader, imdb_valid_loader,
                n_epochs=3, device=device)

Epoch 1/3, train loss: 0.3985, train metric: 0.8417, valid metric: 0.8548
Epoch 2/3, train loss: 0.1048, train metric: 0.9643, valid metric: 0.7658
Epoch 3/3, train loss: 0.0325, train metric: 0.9896, valid metric: 0.8446


In [122]:
# Reuse the entire pretrained BERT model
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True, max_length=200,
                               truncation=True, return_tensors="pt")
bert_output = bert_model(**bert_encoding)
bert_output.last_hidden_state.shape    # contextualized embeddings               

torch.Size([3, 200, 768])

In [123]:
del_vars(["bert_model"])

In [ ]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        embed_dim: int = self.bert.config.hidden_size
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        contextualized_embeddings = self.bert(**encodings).last_hidden_state
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(contextualized_embeddings, lengths=lengths.cpu(),
                              enforce_sorted=False, batch_first=True)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])


In [126]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisModelBert().to(device)
imdb_model_bert.bert.requires_grad_(False)

n_epochs = 4
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train_with_early_stopping(imdb_model_bert, optimizer, xentropy, accuracy,
                imbdb_train_loader, imdb_valid_loader, n_epochs, device=device)

Epoch 1/4, train loss: 0.4860, train metric: 0.7567, valid metric: 0.8702
Epoch 2/4, train loss: 0.3024, train metric: 0.8701, valid metric: 0.8880
Epoch 3/4, train loss: 0.2728, train metric: 0.8868, valid metric: 0.8760
Epoch 4/4, train loss: 0.2531, train metric: 0.8946, valid metric: 0.8668


In [127]:
del_vars(["imdb_model_bert"])

In [128]:
# use only the embedding for the [CLS] token
class SentimentAnalysisModelCLS(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.last_hidden_state[:, 0])

In [129]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisModelCLS().to(device)
imdb_model_bert.bert.requires_grad_(False)

n_epochs = 4
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

history = train_with_early_stopping(imdb_model_bert, optimizer, xentropy, accuracy,
                imbdb_train_loader, imdb_valid_loader, n_epochs, device=device)

Epoch 1/4, train loss: 0.5636, train metric: 0.7346, valid metric: 0.8016
Epoch 2/4, train loss: 0.4640, train metric: 0.7920, valid metric: 0.8138
Epoch 3/4, train loss: 0.4344, train metric: 0.8067, valid metric: 0.8174
Epoch 4/4, train loss: 0.4267, train metric: 0.8097, valid metric: 0.8228


In [131]:
del_vars(["imdb_model_bert"])

#### Task-Specific Classes

In [130]:
from transformers import BertForSequenceClassification

torch.manual_seed(42)
bert_for_binary_clf = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2, dtype=torch.float16).to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [132]:
# Warning: Classification head is still untrained!
encoding = bert_tokenizer(["This was a great movie!"])
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask=torch.tensor(encoding["attention_mask"])
    )
output.logits

tensor([[ 0.0865, -0.0561]], device='mps:0', dtype=torch.float16)

In [133]:
torch.softmax(output.logits, dim=-1)

tensor([[0.5356, 0.4644]], device='mps:0', dtype=torch.float16)

In [134]:
# Returns also the loss if labels are given
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask=torch.tensor(encoding["attention_mask"]),
        labels=torch.tensor([1], device=device)
    )
output.loss

tensor(0.7671, device='mps:0', dtype=torch.float16)

#### The Trainer API

In [135]:
# The API expects datasets of tokenized text
def tokenize_batch(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=200)

tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set = imdb_test_set.map(tokenize_batch, batched=True)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [136]:
def compute_accuracy(pred):
    return {"accuracy": (pred.label_ids == pred.predictions.argmax(-1)).mean()}

In [137]:
from transformers import TrainingArguments

train_args = TrainingArguments(
    output_dir="my_imdb_model", num_train_epochs=4,
    per_device_train_batch_size=256, per_device_eval_batch_size=256,
    eval_strategy="epoch", logging_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    report_to="none"
)

In [138]:
from transformers import DataCollatorWithPadding, Trainer

trainer = Trainer(
    bert_for_binary_clf, train_args, train_dataset=tok_imdb_train_set,
    eval_dataset=tok_imdb_valid_set, compute_metrics=compute_accuracy,
    data_collator=DataCollatorWithPadding(bert_tokenizer)
)

train_output = trainer.train()

/Users/nikolaoschachampis/gitrepos/handson-mlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.364700,nan,0.498800
2,0.000000,nan,0.498800
3,0.000000,nan,0.498800
4,0.000000,nan,0.498800


/Users/nikolaoschachampis/gitrepos/handson-mlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/nikolaoschachampis/gitrepos/handson-mlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/nikolaoschachampis/gitrepos/handson-mlp/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Need to investigate why training didn't work

In [139]:
del_vars(["bert_for_binary_clf"])

#### Pipelines

In [140]:
from transformers import pipeline

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model=model_name,
                           truncation=True, max_length=512)
classifier_imdb(train_reviews[:10])


Device set to use mps:0


[{'label': 'POSITIVE', 'score': 0.9996108412742615},
 {'label': 'POSITIVE', 'score': 0.9998623132705688},
 {'label': 'NEGATIVE', 'score': 0.9943684935569763},
 {'label': 'POSITIVE', 'score': 0.997913658618927},
 {'label': 'POSITIVE', 'score': 0.999544084072113},
 {'label': 'NEGATIVE', 'score': 0.9845332503318787},
 {'label': 'POSITIVE', 'score': 0.9859278202056885},
 {'label': 'POSITIVE', 'score': 0.9993758797645569},
 {'label': 'POSITIVE', 'score': 0.9978922009468079},
 {'label': 'NEGATIVE', 'score': 0.9997020363807678}]

In [141]:
classifier_imdb(["I am from Greece"])

[{'label': 'POSITIVE', 'score': 0.9409297704696655}]

In [142]:
# Text classification
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"
classifier_mnli = pipeline("text-classification", model=model_name)
classifier_mnli([
    "She loves me. [SEP] She loves me not. [SEP]",
    "Alice just woke up. [SEP] Alice is awake. [SEP]",
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"
])

Device set to use mps:0


[{'label': 'contradiction', 'score': 0.9717152714729309},
 {'label': 'entailment', 'score': 0.9119167327880859},
 {'label': 'neutral', 'score': 0.9509281516075134}]